# Handwritten Digit Recognizer — SVM + HOG (all-in-one notebook)

Everything the project needs lives in this one notebook: preprocessing,
loading the trained SVM+HOG model, the live drawing demo.


## 1. Imports

In [14]:
import csv
from pathlib import Path

import cv2
import numpy as np
import joblib
from skimage.feature import hog

print("Imports OK")
print("OpenCV version:", cv2.__version__)

Imports OK
OpenCV version: 5.0.0


## 2. Paths


In [15]:
PROJECT_ROOT = Path.cwd()

MODELS_DIR = PROJECT_ROOT / "models"
SVM_MODEL_PATH = MODELS_DIR / "svm_hog_model.pkl"

DATA_DIR = PROJECT_ROOT / "data"
SAMPLES_DIR = DATA_DIR / "samples"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
SAMPLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Model exists?:", SVM_MODEL_PATH.exists(), "->", SVM_MODEL_PATH)

if not SVM_MODEL_PATH.exists():
    print()
    print("!! svm_hog_model.pkl was not found at the path above.")
    print("!! Move this notebook into the digit-recognizer project folder")
    print("!! (the one that directly contains the 'models' folder) and re-run this cell.")

Project root : C:\Users\M S I\OneDrive\Desktop\digit recognizer\digit-recognizer
Model exists?: True -> C:\Users\M S I\OneDrive\Desktop\digit recognizer\digit-recognizer\models\svm_hog_model.pkl


## 3. Preprocessing

Turns a raw canvas/photo into a centered 28x28 digit image, matching what the SVM was trained on.

In [16]:
def _center_by_mass(image: np.ndarray) -> np.ndarray:
    """Re-center by center of mass, not just bounding box - this is how
    the original MNIST dataset itself was constructed. Skipping it is a
    common reason live-drawn digits get misclassified."""
    moments = cv2.moments(image)
    if moments["m00"] == 0:
        return image

    cx = moments["m10"] / moments["m00"]
    cy = moments["m01"] / moments["m00"]

    shift_x = int(round(image.shape[1] / 2.0 - cx))
    shift_y = int(round(image.shape[0] / 2.0 - cy))

    shift_matrix = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    return cv2.warpAffine(image, shift_matrix, (image.shape[1], image.shape[0]))


def preprocess_image(canvas: np.ndarray):
    """
    Parameters
    ----------
    canvas : np.ndarray
        BGR (or already-grayscale) image containing a white digit on a
        black background.

    Returns
    -------
    np.ndarray of shape (28, 28), dtype uint8, or None if no digit was found.
    """
    if canvas.ndim == 3:
        gray = cv2.cvtColor(canvas, cv2.COLOR_BGR2GRAY)
    else:
        gray = canvas

    _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY)

    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if len(contours) == 0:
        return None

    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    if w == 0 or h == 0:
        return None

    pad = max(2, int(0.05 * max(w, h)))
    x0, y0 = max(x - pad, 0), max(y - pad, 0)
    x1 = min(x + w + pad, thresh.shape[1])
    y1 = min(y + h + pad, thresh.shape[0])
    digit = thresh[y0:y1, x0:x1]

    h, w = digit.shape
    if h > w:
        new_h = 20
        new_w = max(1, int(w * 20 / h))
    else:
        new_w = 20
        new_h = max(1, int(h * 20 / w))

    digit = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)
    digit = cv2.GaussianBlur(digit, (3, 3), 0)

    output = np.zeros((28, 28), dtype=np.uint8)
    x_offset = (28 - new_w) // 2
    y_offset = (28 - new_h) // 2
    output[y_offset:y_offset + new_h, x_offset:x_offset + new_w] = digit

    output = _center_by_mass(output)
    return output

print("preprocess_image() ready")

preprocess_image() ready


## 4. Load the SVM+HOG model and define prediction

In [17]:
HOG_PARAMS = dict(
    orientations=9,
    pixels_per_cell=(4, 4),
    cells_per_block=(2, 2),
    block_norm="L2-Hys",
)

svm_model = joblib.load(SVM_MODEL_PATH)
print("Loaded model:", type(svm_model))


def _extract_hog(image: np.ndarray) -> np.ndarray:
    return hog(image, **HOG_PARAMS)


def predict_digit(processed_image: np.ndarray):
    """
    Returns
    -------
    (digit, confidence) : (int, float | None)
        confidence is None because this SVC was trained without
        probability=True, so no calibrated probability is available.
    """
    features = _extract_hog(processed_image).reshape(1, -1)
    digit = int(svm_model.predict(features)[0])

    confidence = None
    if hasattr(svm_model, "predict_proba"):
        confidence = float(np.max(svm_model.predict_proba(features)))

    return digit, confidence

print("predict_digit() ready")

Loaded model: <class 'sklearn.svm._classes.SVC'>
predict_digit() ready


## 5. Live drawing demo

Running this cell opens a **separate window** on your desktop (not inside
the browser) — Jupyter just triggers it, OpenCV renders its own window.

**Controls**
- Draw: hold the left mouse button and move
- `c` : clear canvas
- `p` : predict
- `Esc` : close the window and return control to the notebook

In [18]:
def run_live_draw():
    CANVAS_SIZE = 600
    BRUSH_THICKNESS = 9  # thick loops (6, 8, 9, 0) can fill in and look like solid blobs if this is too high

    canvas = np.zeros((CANVAS_SIZE, CANVAS_SIZE, 3), dtype=np.uint8)
    state = {"drawing": False, "last_x": -1, "last_y": -1}

    def _on_mouse(event, x, y, flags, param):
        if event == cv2.EVENT_LBUTTONDOWN:
            state["drawing"], state["last_x"], state["last_y"] = True, x, y
        elif event == cv2.EVENT_MOUSEMOVE and state["drawing"]:
            cv2.line(canvas, (state["last_x"], state["last_y"]), (x, y),
                      (255, 255, 255), BRUSH_THICKNESS, lineType=cv2.LINE_AA)
            state["last_x"], state["last_y"] = x, y
        elif event == cv2.EVENT_LBUTTONUP:
            state["drawing"] = False

    def _predict_and_show():
        processed = preprocess_image(canvas)
        if processed is None:
            print("Please draw a digit first!")
            return
        cv2.imshow("Processed 28x28", cv2.resize(processed, (280, 280), interpolation=cv2.INTER_NEAREST))
        digit, confidence = predict_digit(processed)
        print("=" * 40)
        print(f"Predicted Digit : {digit}")
        if confidence is not None:
            print(f"Confidence      : {confidence * 100:.2f}%")
        else:
            print("Confidence      : n/a (model has no probability output)")
        print("=" * 40)

    window = "Handwritten Digit Recognizer (SVM + HOG)"
    cv2.namedWindow(window)
    cv2.setMouseCallback(window, _on_mouse)

    print("Draw a digit, then press: c=clear  p=predict  Esc=quit")

    while True:
        cv2.imshow(window, canvas)
        key = cv2.waitKey(1) & 0xFF
        if key == ord("c"):
            canvas[:] = 0
        elif key == ord("p"):
            _predict_and_show()
        elif key == 27:  # ESC
            break

    cv2.destroyAllWindows()
    cv2.waitKey(1)  # flush window close events on some platforms


run_live_draw()

Draw a digit, then press: c=clear  p=predict  Esc=quit
